In [ ]:
# Clone the OPEN-RAG repository
!git clone https://github.com/ShayekhBinIslam/openrag.git
%cd openrag

# Install the required libraries for quantized inference
!pip install -q transformers peft accelerate bitsandbytes datasets sentencepiece

In [ ]:
from huggingface_hub import notebook_login

# This will prompt you to paste your Hugging Face Access Token
notebook_login()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Define the models (Using the 650MB micro-sharded version)
base_model_id = "Xilabs/Llama-2-7b-Sharded"
adapter_id = "shayekh/openrag_llama2_7b_8x135m_adapters"

# 1. Setup 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Tokenizer with special reflection tokens...")
tokenizer = AutoTokenizer.from_pretrained(adapter_id)

print("Loading Base Llama-2 Model in 650MB chunks (This will take a minute)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True # Forces strict CPU RAM management
)

print("Resizing model embeddings to fit special tokens...")
# RESIZE THE BASE MODEL TO 32,016
base_model.resize_token_embeddings(len(tokenizer))

print("Injecting OPEN-RAG MoE Adapters...")
# 3. Apply the OPEN-RAG PEFT adapters
model = PeftModel.from_pretrained(base_model, adapter_id)

print("Model successfully loaded into VRAM!")

In [ ]:
# Cell 4: Multi-Hop Reasoning Inference Test
# We will use a classic 2-hop setup:
# Hop 1: Who founded the company? (Mohan Singh Oberoi -> Oberoi Group)
# Hop 2: Where is that company's head office? (Oberoi Group -> Delhi)

# 1. The exact instruction format from the OPEN-RAG paper
instruction_text = (
    "You are a question answering agent. Given a context and a question, "
    "your task is to answer the question based on the context. Instead of "
    "a full sentence, your answer must be the shortest word or phrase "
    "or named entity. Some example outputs 'answer' are: yes; no; Ibn Sina; "
    "Doha, Qatar; 2,132 seats, Los Angeles, California etc."
)

# 2. Simulated Multi-Hop Retrieved Context
context = (
    "Knowledge 1: The Oberoi family is an Indian family that is famous for its involvement in hotels. "
    "The family's patriarch, Mohan Singh Oberoi, established the Oberoi Group.\n"
    "Knowledge 2: The Oberoi Group is a hotel company with its head office in Delhi. "
    "Founded in 1934, the company operates 31 luxury hotels."
)

# 3. The Multi-Hop Question
question = "In what city is the head office of the company founded by Mohan Singh Oberoi located?"

# 4. Constructing the final prompt
prompt = f"### Instruction\n{instruction_text}\n\n{context}\n\n{question}\n### Response:\n"

print("Tokenizing prompt...")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating response (Greedy Decoding)...")
# We use do_sample=False to mimic their greedy decoding inference setup
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

print("\n--- MODEL OUTPUT ---")
# We keep skip_special_tokens=False so you can see OPEN-RAG's internal "reflection" tokens!
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
print(response)